# Retirement Planning Simulation

Long-horizon wealth simulation across multiple accounts — brokerage, 401k, Roth IRA, 529s, HSA, and real estate.

**How to use this notebook**
1. Update the parameters in sections 1–5 to match your situation.
2. Run all cells top-to-bottom.
3. Explore the fan charts, per-account trajectories, and scenario analysis at the end.

In [1]:
import waypoint as wp
import numpy as np
import polars as pl
import plotly.graph_objects as go
import plotly.express as px

## 1. Inflation — pick a rate

Fetch CPI history and plot it to inform the inflation assumption used throughout the simulation.

In [2]:
# CPI_YOY is an AssetDef (monthly pct-change of CPIAUCSL).
# Rolling 12-month sum approximates YoY inflation.
cpi_asset = wp.fetch(wp.catalog.fixed_income.CPI_YOY, start="2000-01-01", end="2025-12-31")

cpi_yoy = (
    cpi_asset.returns
    .with_columns(
        (pl.col("returns").rolling_sum(window_size=12) * 100).alias("cpi_yoy_pct")
    )
    .drop_nulls()
)

fig = px.line(
    cpi_yoy.to_pandas(),
    x="date",
    y="cpi_yoy_pct",
    title="US CPI Year-over-Year (12-month rolling, %)",
    labels={"cpi_yoy_pct": "CPI YoY (%)", "date": ""},
)
fig.add_hline(y=2.5, line_dash="dash", line_color="gray", annotation_text="2.5% baseline")
fig.show()

In [3]:
# ── Tune these ─────────────────────────────────────────────────────────────
INFLATION_RATE = 0.03   # annual; used for all real-terms cashflows
RETURN_METHOD  = wp.returns.ShrinkageTowardGrandMean(alpha=0.75)
RISK_METHOD    = wp.risk.LedoitWolf()
# ───────────────────────────────────────────────────────────────────────────

## 2. Timeline constants

Set your current age, target retirement age, and planning horizon.
If you have children, set their ages to size the 529 contribution windows.

In [4]:
# ── Tune these ─────────────────────────────────────────────────────────────
CURRENT_AGE      = 35    # your current age
RETIREMENT_AGE   = 60    # target retirement age
HORIZON_AGE      = 90    # planning horizon (age, not years)
BIRTH_YEAR       = 2026 - CURRENT_AGE  # derived; update CURRENT_AGE instead

# Children — set to None if no children or no 529 accounts
KID1_AGE         = 4  # e.g. 5 → college starts in 13 years
KID2_AGE         = 7  # e.g. 3 → college starts in 15 years

# Whether to express output wealth in real (inflation-adjusted) terms
WEALTH_INFLATION_ADJUSTED = True

# Whether contributions grow with inflation
CONTRIBS_INFLATION_ADJUSTED = False
# ───────────────────────────────────────────────────────────────────────────

HORIZON_YEARS   = HORIZON_AGE - CURRENT_AGE
RETIREMENT_YEAR = RETIREMENT_AGE - CURRENT_AGE

if KID1_AGE is not None:
    KID1_COLLEGE_START = 18 - KID1_AGE
    KID1_COLLEGE_END   = KID1_COLLEGE_START + 4
    print(f"Kid 1 college: years {KID1_COLLEGE_START}–{KID1_COLLEGE_END}")

if KID2_AGE is not None:
    KID2_COLLEGE_START = 18 - KID2_AGE
    KID2_COLLEGE_END   = KID2_COLLEGE_START + 4
    print(f"Kid 2 college: years {KID2_COLLEGE_START}–{KID2_COLLEGE_END}")

print(f"Horizon: {HORIZON_YEARS} years  |  Retirement year: {RETIREMENT_YEAR}")

Kid 1 college: years 14–18
Kid 2 college: years 11–15
Horizon: 55 years  |  Retirement year: 25


## 3. Historical data

Fetch returns for all asset classes used across the accounts.
Adjust `DATA_START` / `DATA_END` to control the estimation window.

In [5]:
# ── Tune these ─────────────────────────────────────────────────────────────
DATA_START = "2005-01-01"
DATA_END   = "2025-12-31"
# ───────────────────────────────────────────────────────────────────────────

nasdaq100     = wp.fetch(wp.catalog.equities.NASDAQ_100,           start=DATA_START, end=DATA_END)
russell1000   = wp.fetch(wp.catalog.equities.RUSSELL_1000,         start=DATA_START, end=DATA_END)
europe        = wp.fetch(wp.catalog.equities.EUROPE_DEVELOPED,     start=DATA_START, end=DATA_END)
china_tech    = wp.fetch(wp.catalog.equities.CHINA_TECH,           start=DATA_START, end=DATA_END)
financials    = wp.fetch(wp.catalog.equities.US_FINANCIALS,        start=DATA_START, end=DATA_END)
us_total      = wp.fetch(wp.catalog.equities.US_TOTAL_MARKET,      start=DATA_START, end=DATA_END)
us_lcg        = wp.fetch(wp.catalog.equities.US_LARGE_CAP_GROWTH,  start=DATA_START, end=DATA_END)
tbills        = wp.fetch(wp.catalog.fixed_income.RISK_FREE_RATE,   start=DATA_START, end=DATA_END)
us_agg_bonds  = wp.fetch(wp.catalog.fixed_income.US_AGG_BONDS,     start=DATA_START, end=DATA_END)

# Real estate index — only needed if you include a real estate account below
# Replace with a region-appropriate HPI from wp.catalog.real_estate
local_hpi     = wp.fetch(wp.catalog.real_estate.BOSTON_HPI,        start=DATA_START, end=DATA_END)

print("Data loaded.")

$^NDX: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-01-03)
$^RUI: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-01-03)
$VGK: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-03-10) (Yahoo error = "Data doesn't exist for startDate = 1104555600, endDate = 1110430800")
$CQQQ: possibly delisted; no price data found  (1d 2005-01-01 -> 2010-01-22) (Yahoo error = "Data doesn't exist for startDate = 1104555600, endDate = 1264136400")
$XLF: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-01-03)
$VTI: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-01-03)
$VUG: possibly delisted; no price data found  (1d 2005-01-01 -> 2005-01-03)


Data loaded.


## 4. Social Security estimate

Estimate your Social Security benefit at various claiming ages.
`ANNUAL_SALARY` and `CAREER_YEARS` are used to approximate your AIME.

In [6]:
# ── Tune these ─────────────────────────────────────────────────────────────
ANNUAL_SALARY    = 120_000   # representative current (or average) earnings
CAREER_YEARS     = 35        # years of earnings history (approximate)
SS_CLAIM_AGE     = 67.0      # full retirement age for most birth years 1960+
# ───────────────────────────────────────────────────────────────────────────

aime = wp.social_security.estimate_aime(ANNUAL_SALARY, career_years=CAREER_YEARS)
fra  = wp.social_security.full_retirement_age(BIRTH_YEAR)

print(f"Estimated AIME:  ${aime:,.2f}/month")
print(f"Full Retirement Age: {fra}")
print()

for claim_age in [62, 67, 70]:
    benefit = wp.social_security.monthly_benefit(aime, BIRTH_YEAR, float(claim_age))
    print(f"  Claim at {claim_age}: ${benefit:,.0f}/month  (${benefit * 12:,.0f}/year)")

Estimated AIME:  $10,000.00/month
Full Retirement Age: 67.0

  Claim at 62: $2,369/month  ($28,427/year)
  Claim at 67: $3,384/month  ($40,610/year)
  Claim at 70: $4,196/month  ($50,357/year)


In [7]:
# Simulation year when SS payments begin = claim age − current age
SS_START_YEAR = SS_CLAIM_AGE - CURRENT_AGE

ss_cashflow = wp.social_security.as_cashflow(
    aime=aime,
    birth_year=BIRTH_YEAR,
    claim_age=SS_CLAIM_AGE,
    real=True,            # COLA-adjusted in real terms
    effective_tax_rate=0.0,
    start_year=SS_START_YEAR,
)

print(f"SS cashflow: ${ss_cashflow.amount:,.2f}/month starting year {SS_START_YEAR}")

SS cashflow: $3,384.18/month starting year 32.0


## 5. Build portfolios

Each account is a `wp.Portfolio` with its own asset allocation, initial wealth, and cashflow list.
Adjust balances, allocations, and contribution amounts to match your situation.
Remove or comment out account sections that don't apply.

### 5a. Brokerage account

- Contributions flow to equity slots pre-retirement
- Retirement spending drawn from all slots post-retirement
- Social Security income deposited here from SS start year

In [8]:
# ── Tune these ─────────────────────────────────────────────────────────────
BROKERAGE_INITIAL_WEALTH  = 100_000.0   # current balance
BROKERAGE_ANNUAL_CONTRIB  =  20_000.0   # annual savings (pre-retirement)
ANNUAL_RETIREMENT_SPENDING = 80_000.0   # real spending draw post-retirement
CAP_GAINS_TAX_RATE         = 0.18       # effective cap-gains tax on withdrawals

# Optional: net rental income from a property you own
MONTHLY_NET_RENT = 0.0   # set to 0 if no rental property
# ───────────────────────────────────────────────────────────────────────────

BROKERAGE_EQUITY_SLOTS = ("nasdaq100", "russell1000", "europe", "financials")

# After-tax spending draw (gross up for cap-gains tax)
ANNUAL_SPENDING_GROSS = ANNUAL_RETIREMENT_SPENDING / (1 - CAP_GAINS_TAX_RATE)

brokerage = wp.Portfolio(
    slots={
        "nasdaq100":    nasdaq100,
        "russell1000":  russell1000,
        "europe":       europe,
        "financials":   financials,
        "tbills":       tbills,
        "us_agg_bonds": us_agg_bonds,
    },
    weights={
        "nasdaq100":    0.20,
        "russell1000":  0.20,
        "europe":       0.10,
        "financials":   0.10,
        "tbills":       0.30,
        "us_agg_bonds": 0.10,
    },
    name="brokerage",
    initial_wealth=BROKERAGE_INITIAL_WEALTH,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

brokerage_cashflows = [
    # Annual contributions (pre-retirement, into equity slots)
    wp.cashflows.PeriodicCashflow(
        amount=BROKERAGE_ANNUAL_CONTRIB,
        frequency="annual",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(RETIREMENT_YEAR),
        slots=BROKERAGE_EQUITY_SLOTS,
    ),
    # Net rental income (optional — set MONTHLY_NET_RENT=0 to disable)
    wp.cashflows.PeriodicCashflow(
        amount=MONTHLY_NET_RENT,
        frequency="monthly",
        real=False,   # nominal — rent and costs roughly offset
    ),
    # Social Security income (COLA-adjusted)
    ss_cashflow,
    # Retirement spending (real, post-retirement)
    wp.cashflows.PeriodicCashflow(
        amount=-ANNUAL_SPENDING_GROSS,
        frequency="annual",
        real=True,
        start_year=float(RETIREMENT_YEAR),
    ),
]

### 5b. 401(k)

In [9]:
# ── Tune these ─────────────────────────────────────────────────────────────
ACCOUNT_401K_BALANCE       = 200_000.0   # current balance
MONTHLY_401K_CONTRIB       =   1_500.0   # your monthly employee contribution
ANNUAL_EMPLOYER_CONTRIB    =       0.0   # employer match / profit-sharing (annual)
# ───────────────────────────────────────────────────────────────────────────

portfolio_401k = wp.Portfolio(
    slots={
        "us_total":     us_total,
        "us_lcg":       us_lcg,
        "us_agg_bonds": us_agg_bonds,
    },
    weights={"us_total": 0.50, "us_lcg": 0.50, "us_agg_bonds": 0.0},
    name="401k",
    initial_wealth=ACCOUNT_401K_BALANCE,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

cashflows_401k = [
    wp.cashflows.PeriodicCashflow(
        amount=MONTHLY_401K_CONTRIB,
        frequency="monthly",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(RETIREMENT_YEAR),
    ),
    wp.cashflows.PeriodicCashflow(
        amount=ANNUAL_EMPLOYER_CONTRIB,
        frequency="annual",
        real=False,
        end_year=float(RETIREMENT_YEAR),
    ),
]

### 5c. Roth IRA

In [10]:
# ── Tune these ─────────────────────────────────────────────────────────────
ROTH_BALANCE      = 10_000.0   # current balance
ANNUAL_ROTH_CONTRIB = 7_000.0  # IRS limit (2025); adjust each year
# ───────────────────────────────────────────────────────────────────────────

roth_ira = wp.Portfolio(
    slots={"us_total": us_total, "us_agg_bonds": us_agg_bonds},
    weights={"us_total": 1.0, "us_agg_bonds": 0.0},
    name="roth_ira",
    initial_wealth=ROTH_BALANCE,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

cashflows_roth = [
    wp.cashflows.PeriodicCashflow(
        amount=ANNUAL_ROTH_CONTRIB,
        frequency="annual",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(RETIREMENT_YEAR),
    ),
]

### 5d. 529 — Child 1

Skip this section (and remove from the `wp.Aggregate` below) if you have no 529 accounts.

In [11]:
# ── Tune these ─────────────────────────────────────────────────────────────
CHILD1_529_BALANCE   =  5_000.0  # current balance
CHILD1_MONTHLY_CONTRIB = 300.0   # monthly savings until college
# ───────────────────────────────────────────────────────────────────────────

# Requires KID1_AGE set above
if KID1_AGE is None:
    raise ValueError("Set KID1_AGE in section 2 before running this cell")

portfolio_529_1 = wp.Portfolio(
    slots={"us_total": us_total, "us_agg_bonds": us_agg_bonds},
    weights={"us_total": 1.0, "us_agg_bonds": 0.0},
    name="529_child1",
    initial_wealth=CHILD1_529_BALANCE,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

cashflows_529_1 = [
    wp.cashflows.PeriodicCashflow(
        amount=CHILD1_MONTHLY_CONTRIB,
        frequency="monthly",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(KID1_COLLEGE_START),
    ),
    # Withdraw ~25% per year over 4 college years
    wp.cashflows.PeriodicCashflow(
        amount=-0.25,
        frequency="annual",
        mode="pct_portfolio",
        start_year=float(KID1_COLLEGE_START),
        end_year=float(KID1_COLLEGE_END),
    ),
]

### 5e. 529 — Child 2

Skip this section if you have fewer than two children.

In [12]:
# ── Tune these ─────────────────────────────────────────────────────────────
CHILD2_529_BALANCE    = 1_000.0  # current balance
CHILD2_MONTHLY_CONTRIB = 300.0   # monthly savings until college
# ───────────────────────────────────────────────────────────────────────────

if KID2_AGE is None:
    raise ValueError("Set KID2_AGE in section 2 before running this cell")

portfolio_529_2 = wp.Portfolio(
    slots={"us_total": us_total, "us_agg_bonds": us_agg_bonds},
    weights={"us_total": 1.0, "us_agg_bonds": 0.0},
    name="529_child2",
    initial_wealth=CHILD2_529_BALANCE,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

cashflows_529_2 = [
    wp.cashflows.PeriodicCashflow(
        amount=CHILD2_MONTHLY_CONTRIB,
        frequency="monthly",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(KID2_COLLEGE_START),
    ),
    wp.cashflows.PeriodicCashflow(
        amount=-0.25,
        frequency="annual",
        mode="pct_portfolio",
        start_year=float(KID2_COLLEGE_START),
        end_year=float(KID2_COLLEGE_END),
    ),
]

### 5f. HSA

In [13]:
# ── Tune these ─────────────────────────────────────────────────────────────
HSA_BALANCE          = 10_000.0  # current balance
MONTHLY_HSA_CONTRIB  =    500.0  # monthly contribution until retirement
# ───────────────────────────────────────────────────────────────────────────

hsa = wp.Portfolio(
    slots={"us_total": us_total, "us_agg_bonds": us_agg_bonds},
    weights={"us_total": 1.0, "us_agg_bonds": 0.0},
    name="hsa",
    initial_wealth=HSA_BALANCE,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

cashflows_hsa = [
    wp.cashflows.PeriodicCashflow(
        amount=MONTHLY_HSA_CONTRIB,
        frequency="monthly",
        real=CONTRIBS_INFLATION_ADJUSTED,
        end_year=float(RETIREMENT_YEAR),
    ),
]

### 5g. Real estate (optional)

Model a leveraged property position using `wp.LeveragedAsset`.
Net rental income is modeled as a cashflow in the brokerage account above.
Remove this section and exclude `real_estate` from `wp.Aggregate` if not applicable.

In [14]:
# ── Tune these ─────────────────────────────────────────────────────────────
PROPERTY_VALUE   = 500_000.0   # current market value
MORTGAGE_BALANCE = 300_000.0   # outstanding mortgage principal
MORTGAGE_RATE    = 0.065       # annual interest rate
# ───────────────────────────────────────────────────────────────────────────

PROPERTY_EQUITY  = PROPERTY_VALUE - MORTGAGE_BALANCE
LEVERAGE_RATIO   = PROPERTY_VALUE / PROPERTY_EQUITY

condo_asset = wp.LeveragedAsset(
    asset=local_hpi,
    leverage_ratio=LEVERAGE_RATIO,
    financing_cost=MORTGAGE_RATE,
    name="Property (Leveraged)",
)

real_estate = wp.Portfolio(
    slots={"property": condo_asset, "us_agg_bonds": us_agg_bonds},
    weights={"property": 1.0, "us_agg_bonds": 0.0},
    name="real_estate",
    initial_wealth=PROPERTY_EQUITY,
    expected_return_method=RETURN_METHOD,
    risk_method=RISK_METHOD,
)

# Net rental income cashflow is in brokerage_cashflows (MONTHLY_NET_RENT above)
cashflows_real_estate: list = []

print(f"Property equity: ${PROPERTY_EQUITY:,.0f}")
print(f"Leverage ratio:  {LEVERAGE_RATIO:.2f}x")

Property equity: $200,000
Leverage ratio:  2.50x


## 6. Aggregate all accounts

Combine all portfolios into a single `wp.Aggregate`. Remove accounts that don't apply.

In [15]:
aggregate = wp.Aggregate([
    brokerage,
    portfolio_401k,
    roth_ira,
    portfolio_529_1,   # remove if no children / 529s
    portfolio_529_2,   # remove if fewer than 2 children
    hsa,
    real_estate,       # remove if no property
])

total_wealth = sum(p.initial_wealth for p in aggregate.portfolios)
weights = aggregate.wealth_weights()

print(f"Total initial wealth: ${total_wealth:,.0f}")
print()
for name, w in weights.items():
    acct = next(p for p in aggregate.portfolios if p.name == name)
    print(f"  {name:<20} ${acct.initial_wealth:>12,.0f}   ({w:.1%})")

Total initial wealth: $526,000

  brokerage            $     100,000   (19.0%)
  401k                 $     200,000   (38.0%)
  roth_ira             $      10,000   (1.9%)
  529_child1           $       5,000   (1.0%)
  529_child2           $       1,000   (0.2%)
  hsa                  $      10,000   (1.9%)
  real_estate          $     200,000   (38.0%)


### 6b. Asset risk & return

Estimated using the wealth-weighted flattened aggregate, quarterly frequency.

In [16]:
_flat = aggregate.flatten()

_er = wp.analytics.ExpectedReturn(method=RETURN_METHOD).compute(
    _flat, start=DATA_START, end=DATA_END, frequency="quarterly"
)
_risk = wp.analytics.Risk(method=RISK_METHOD).compute(
    _flat, start=DATA_START, end=DATA_END, frequency="quarterly"
)

_risk.plot_risk_return(_er).show()

In [17]:
_risk.plot().show()

### 6c. Efficient frontier

Mean-variance frontier for the flattened aggregate. Constraints: long-only, weights sum to 1.

In [18]:
RF_RATE = _er.per_asset["tbills"]
print(f"Risk-free rate (T-bill, annualised): {RF_RATE:.2%}")

opt = wp.analytics.Optimizer(
    return_model=wp.analytics.ExpectedReturn(method=RETURN_METHOD),
    risk_model=wp.analytics.Risk(method=RISK_METHOD),
    constraints=[wp.LongOnly(), wp.SumToOne(), wp.WeightBounds(max_weight=0.6)],
)
frontier = opt.efficient_frontier(_flat, start=DATA_START, end=DATA_END, frequency="quarterly")
frontier.plot().show()

# Max Sharpe portfolio
sharpe_weights = frontier.optimal_sharpe(RF_RATE)
rets = frontier.expected_returns.to_list()
vols = frontier.risks.to_list()
sharpe_ratios = [(r - RF_RATE) / v if v > 0 else float("-inf") for r, v in zip(rets, vols)]
sharpe_idx = max(range(len(sharpe_ratios)), key=lambda i: sharpe_ratios[i])

print(f"\nMax Sharpe  (return {rets[sharpe_idx]:.2%}, vol {vols[sharpe_idx]:.2%}, Sharpe {sharpe_ratios[sharpe_idx]:.2f})")
for name, w in sorted(sharpe_weights.items(), key=lambda kv: -kv[1]):
    if w > 0.005:
        print(f"  {name:<20} {w:.1%}")

Risk-free rate (T-bill, annualised): 6.23%



Max Sharpe  (return 8.08%, vol 8.87%, Sharpe 0.21)
  us_agg_bonds         57.0%
  nasdaq100            43.0%


## 7. Simulation

Run `MultiWealthSimulation` across all accounts.
Boston HPI is quarterly so all assets are downsampled to quarterly frequency.

In [19]:
# ── Tune these ─────────────────────────────────────────────────────────────
N_SIMULATIONS = 2_000
# ───────────────────────────────────────────────────────────────────────────

sim = wp.analytics.MultiWealthSimulation(
    method=wp.sim.MonteCarlo(),
    cashflows={
        "brokerage":    brokerage_cashflows,
        "401k":         cashflows_401k,
        "roth_ira":     cashflows_roth,
        "529_child1":   cashflows_529_1,   # remove if no children
        "529_child2":   cashflows_529_2,   # remove if fewer than 2 children
        "hsa":          cashflows_hsa,
        "real_estate":  cashflows_real_estate,  # remove if no property
    },
    horizon_years=HORIZON_YEARS,
    n_simulations=N_SIMULATIONS,
    inflation_rate=INFLATION_RATE,
)

result = sim.compute(
    aggregate,
    start=DATA_START,
    end=DATA_END,
    frequency="quarterly",
    real=WEALTH_INFLATION_ADJUSTED,
)

print("Simulation complete.")

Simulation complete.


## 8. Annual cashflow schedule

Net inflows (+) and outflows (−) per account each year, in real dollars.

In [20]:
print(result.cashflow_schedule)
result.plot_cashflow_schedule().show()

shape: (55, 9)
┌──────┬────────────┬────────────┬────────────┬───┬────────────┬───────────┬───────────┬───────────┐
│ year ┆ brokerage  ┆ 401k       ┆ roth_ira   ┆ … ┆ 529_child2 ┆ hsa       ┆ real_esta ┆ total     │
│ ---  ┆ ---        ┆ ---        ┆ ---        ┆   ┆ ---        ┆ ---       ┆ te        ┆ ---       │
│ i64  ┆ f64        ┆ f64        ┆ f64        ┆   ┆ f64        ┆ f64       ┆ ---       ┆ f64       │
│      ┆            ┆            ┆            ┆   ┆            ┆           ┆ f64       ┆           │
╞══════╪════════════╪════════════╪════════════╪═══╪════════════╪═══════════╪═══════════╪═══════════╡
│ 1    ┆ 19417.4757 ┆ 17475.7281 ┆ 6796.11650 ┆ … ┆ 3495.14563 ┆ 5825.2427 ┆ 0.0       ┆ 56504.854 │
│      ┆ 28         ┆ 55         ┆ 5          ┆   ┆ 1          ┆ 18        ┆           ┆ 369       │
│ 2    ┆ 18851.9181 ┆ 16966.7263 ┆ 6598.17136 ┆ … ┆ 3393.34527 ┆ 5655.5754 ┆ 0.0       ┆ 54859.081 │
│      ┆ 83         ┆ 64         ┆ 4          ┆   ┆ 3          ┆ 55        ┆

## 9. Total wealth fan chart

In [21]:
result.total.plot().update_layout(title="Total Net Worth — Fan Chart (Real $)").show()

paths = result.total.paths
print(f"Median terminal wealth:   ${float(np.median(paths[:, -1])):>14,.0f}")
print(f"10th pct terminal wealth: ${float(np.percentile(paths[:, -1], 10)):>14,.0f}")
print(f"25th pct terminal wealth: ${float(np.percentile(paths[:, -1], 25)):>14,.0f}")
print(f"P(ruin):                   {float(np.mean(paths[:, -1] <= 0)):>14.1%}")

Median terminal wealth:   $     7,336,961
10th pct terminal wealth: $       448,055
25th pct terminal wealth: $     2,628,722
P(ruin):                             6.3%


## 10. Per-account median trajectories

In [22]:
result.plot_accounts().show()

## 11. Terminal wealth by account

In [23]:
rows = []
for name, acct_result in result.accounts.items():
    terminal = acct_result.paths[:, -1]
    rows.append({
        "account": name,
        "median": float(np.median(terminal)),
        "p10":    float(np.percentile(terminal, 10)),
        "p90":    float(np.percentile(terminal, 90)),
        "p_ruin": float(np.mean(terminal <= 0)),
    })

summary_df = pl.DataFrame(rows)
print(
    summary_df.with_columns([
        pl.col("median").map_elements(lambda v: f"${v:,.0f}", return_dtype=pl.String),
        pl.col("p10").map_elements(lambda v: f"${v:,.0f}", return_dtype=pl.String),
        pl.col("p90").map_elements(lambda v: f"${v:,.0f}", return_dtype=pl.String),
        pl.col("p_ruin").map_elements(lambda v: f"{v:.1%}", return_dtype=pl.String),
    ])
)

shape: (7, 5)
┌─────────────┬─────────────┬─────────────┬─────────────┬────────┐
│ account     ┆ median      ┆ p10         ┆ p90         ┆ p_ruin │
│ ---         ┆ ---         ┆ ---         ┆ ---         ┆ ---    │
│ str         ┆ str         ┆ str         ┆ str         ┆ str    │
╞═════════════╪═════════════╪═════════════╪═════════════╪════════╡
│ brokerage   ┆ $-1,008,668 ┆ $-3,227,657 ┆ $5,578,985  ┆ 66.4%  │
│ 401k        ┆ $5,517,166  ┆ $1,344,400  ┆ $25,408,364 ┆ 0.0%   │
│ roth_ira    ┆ $1,072,830  ┆ $277,720    ┆ $4,416,324  ┆ 0.0%   │
│ 529_child1  ┆ $105,163    ┆ $25,057     ┆ $456,270    ┆ 0.0%   │
│ 529_child2  ┆ $79,697     ┆ $19,083     ┆ $356,154    ┆ 0.0%   │
│ hsa         ┆ $955,034    ┆ $247,637    ┆ $3,980,066  ┆ 0.0%   │
│ real_estate ┆ $645,289    ┆ $282,041    ┆ $1,416,139  ┆ 0.0%   │
└─────────────┴─────────────┴─────────────┴─────────────┴────────┘


## 12. Scenario: vary Social Security claiming age

Compare total wealth trajectories when claiming SS at 62, 67, and 70.

In [24]:
scenario_results = {}

for claim_age in [62.0, 67.0, 70.0]:
    cf_ss = wp.social_security.as_cashflow(
        aime=aime,
        birth_year=BIRTH_YEAR,
        claim_age=claim_age,
        real=True,
        start_year=claim_age - CURRENT_AGE,
    )
    scenario_brokerage_cfs = [
        brokerage_cashflows[0],  # contributions
        brokerage_cashflows[1],  # rent
        cf_ss,                   # SS at this claim age
        brokerage_cashflows[3],  # retirement spending
    ]
    scenario_sim = wp.analytics.MultiWealthSimulation(
        method=wp.sim.MonteCarlo(),
        cashflows={
            "brokerage":   scenario_brokerage_cfs,
            "401k":        cashflows_401k,
            "roth_ira":    cashflows_roth,
            "529_child1":  cashflows_529_1,
            "529_child2":  cashflows_529_2,
            "hsa":         cashflows_hsa,
            "real_estate": cashflows_real_estate,
        },
        horizon_years=HORIZON_YEARS,
        n_simulations=1_000,
        inflation_rate=INFLATION_RATE,
    )
    r = scenario_sim.compute(
        aggregate, start=DATA_START, end=DATA_END,
        frequency="quarterly", real=WEALTH_INFLATION_ADJUSTED,
    )
    scenario_results[f"SS @ {int(claim_age)}"] = r

print("Scenarios run.")

Scenarios run.


In [25]:
fig = go.Figure()

for label, r in scenario_results.items():
    median_path = np.median(r.total.paths, axis=0)
    fig.add_trace(go.Scatter(
        x=list(range(len(median_path))),
        y=median_path.tolist(),
        name=label,
        mode="lines",
    ))

fig.update_layout(
    title="Total Wealth — SS Claiming Age Scenarios (Median, Real $)",
    xaxis_title="Quarter",
    yaxis_title="Wealth ($)",
)
fig.show()

print("\nTerminal wealth (median, real $):")
for label, r in scenario_results.items():
    med = float(np.median(r.total.paths[:, -1]))
    print(f"  {label}: ${med:>12,.0f}")


Terminal wealth (median, real $):
  SS @ 62: $   7,436,763
  SS @ 67: $   7,284,544
  SS @ 70: $   7,235,979
